# Thomson-1 — Authority Deference

A standalone run of the seven-arm authority experiment against [`thomsonreuters/Thomson-1.0-Small`](https://huggingface.co/thomsonreuters/Thomson-1.0-Small), kept separate from the main sweep because it is much larger and slower than the other models.

Why this model specifically: it is a **shipped legal product's model**, not a research base. Its card gives the base as `tri-fair-lab/Snowdon1.1-Small` and the architecture as `Qwen3_5MoeForConditionalGeneration` — a Qwen3.5 mixture-of-experts derivative. That closes the last inferential step between this project's finding and a system in the field.

**Be open to it passing.** Thomson-1 was post-trained on curated legal data with explicit value alignment (Constitutional DPO, a conformance reward in RL). That may well fix the deference its base family shows. If it does, that is a better and more useful result than if it does not — it would show the failure is real in base models and that targeted post-training addresses it.

## Licence

Thomson-1.0-Small is released under **PolyForm Strict 1.0.0**. The terms permit *"research, experiment, and testing for the benefit of public knowledge"* and use by *"educational institution[s], public research organization[s]"*, and explicitly preserve fair use. They contain **no restriction on benchmarking or on publishing results**.

They do forbid *"distributing the software or making changes or new works based on the software"* — this notebook does neither; it runs inference and records numbers.

The permission attaches to the **purpose** being noncommercial. Publishing an open paper is a noncommercial purpose. Work in service of a commercial product or offering is a different question and is yours to judge.

Full text: <https://polyformproject.org/licenses/strict/1.0.0>

## 1. Repository

In [ ]:
import os, sys, json, pathlib, subprocess

REPO_URL = "https://github.com/ryanmcdonough/behaviour-microscope.git"
REPO_DIR = pathlib.Path("/content/behaviour-microscope")

if not REPO_DIR.exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    dirty = subprocess.run(["git", "-C", str(REPO_DIR), "status", "--porcelain"],
                           capture_output=True, text=True).stdout.strip()
    if dirty:
        print("Local changes — stashing:\n" + dirty)
        !git -C $REPO_DIR stash -u
    !git -C $REPO_DIR fetch --depth 1 origin main -q
    !git -C $REPO_DIR reset --hard origin/main -q

os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
!git -C $REPO_DIR log --oneline -1

## 2. Install

In [ ]:
%pip install -q -e '/content/behaviour-microscope'
print("installed")

## 3. Hardware check

The weights are **70.2 GB**. This needs the A100-**80GB** — the *High RAM* toggle in Runtime → Change runtime type. On the 40GB card the load will fail partway through, which surfaces as a confusing OOM rather than a clear message, so this cell refuses up front.

In [ ]:
import torch

WEIGHTS_GB = 70.2
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."
props = torch.cuda.get_device_properties(0)
total = props.total_memory / 1e9
print(f"{props.name}  |  {total:.1f} GB  |  compute {props.major}.{props.minor}")

headroom = total - WEIGHTS_GB
if headroom < 4:
    raise SystemExit(
        f"This card has {total:.0f} GB; the weights alone are {WEIGHTS_GB} GB.\n"
        "Switch to the A100-80GB (High RAM) runtime.")
print(f"Headroom after weights: ~{headroom:.0f} GB — enough for these short prompts, but not generous.")

DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
DEVICE = "cuda"
free_disk = os.statvfs('/content').f_bavail * os.statvfs('/content').f_frsize / 1e9
print(f"dtype: {DTYPE}   device: {DEVICE}   free disk: {free_disk:.0f} GB (need ~{WEIGHTS_GB:.0f})")

## 4. Compatibility pre-flight

Builds the model skeleton on the *meta* device from its config — no weights, no download — and asks interp-engine whether it can address the points the experiment needs. **Run this before starting a 70 GB download.** If a point fails here, the download would have been wasted.

In [ ]:
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from interp_engine import EagerModel

MODEL_ID = "thomsonreuters/Thomson-1.0-Small"

cfg = AutoConfig.from_pretrained(MODEL_ID)
print("architecture:", (cfg.architectures or ['?'])[0])
with torch.device("meta"):
    skeleton = AutoModelForCausalLM.from_config(cfg)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
probe = EagerModel(MODEL_ID, hf_model=skeleton, tokenizer=tok)

print(f"layers: {probe.n_layers}   d_model: {probe.d_model}   "
      f"residual streams: {probe.residual_basis.n_streams}   lens valid: {probe.residual_basis.lens_valid}")
print()
ok = True
for name in ["resid_post", "resid_pre", "mlp_out", "attn_out", "router_logits"]:
    try:
        mod, side = probe.resolve_point(name, 10)
        print(f"  {name:14s} OK   {type(mod).__name__} ({side})")
    except Exception as e:
        ok = False
        print(f"  {name:14s} FAIL {type(e).__name__}: {e}")

assert ok, "A required point could not be addressed — do not start the download."
print("\nAll required points resolve. resid_post is the one every experiment depends on.")
del probe, skeleton

## 5. Scenarios and arms

Identical to the main experiment — same 30 UK legal scenarios, same seven arms, same answer key. Nothing is tuned for this model.

In [ ]:
from microscope.experiment import RunConfig, run_sweep, compare_runs
from microscope.scenarios import ARMS, load_scenarios
import pandas as pd

scenarios = load_scenarios()
print(len(scenarios), "scenarios")
for arm in ARMS:
    print(f"  {arm.name:20s} {arm.cue or '(no assertion)'}")

## 6. Run

Two configurations, because Thomson-1 is a reasoning model (its chat template reads `enable_thinking` and emits `<think>`):

1. **Reasoning off** — comparable with the gemma and Qwen runs, and the only one that can do the mechanistic experiments (with reasoning on, the answer is no longer at the final prompt position for patching to reach).
2. **Reasoning on** — behavioural only. Closer to how it would actually be deployed.

**This is slow.** A 35B mixture-of-experts model on 210 prompts, plus a 40-layer bidirectional patching sweep. Budget 1–2 hours for the first configuration. Set `MECHANISTIC = False` to skip the patching and get the behavioural numbers in a fraction of the time.

In [ ]:
MECHANISTIC = True     # False -> experiment 1 only, far faster on a 35B MoE
RUN_THINKING = True    # also run the reasoning-on configuration

def thomson(thinking):
    return RunConfig(
        model_id=MODEL_ID, provider="local", backend="eager", dtype=DTYPE,
        extra_load_kwargs={"device": DEVICE}, n_candidate_layers=4,
        enable_thinking=thinking,
        mechanistic=MECHANISTIC,   # a reasoning run is behavioural-only regardless
        # arms=("floor", "junior_said", "partner_said", "partner_confirmed", "court"),
    )

configs = [thomson(thinking=False)]
if RUN_THINKING:
    configs.append(thomson(thinking=True))

import concurrent.futures
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
    runs = pool.submit(run_sweep, configs).result()
runs

## 7. Results

In [ ]:
for label, path in runs.items():
    s = json.loads((path / "summary.json").read_text())
    q = json.loads((path / "quality_report.json").read_text())
    print(f"=== {label} — quality gate: {q['overall'].upper()} ===")
    for arm, rate in s["behavioural"]["fpar_by_arm"].items():
        acc = s["behavioural"]["accuracy_by_arm"][arm]
        print(f"  {arm:20s} accepts false {rate:5.0%}   accuracy {acc:4.0%}")
    print()

In [ ]:
# Against the other models, if their runs are on this machine.
from pathlib import Path
everything = dict(runs)
for p in sorted(Path("results").glob("*Z")):
    if not (p / "summary.json").exists() or p in runs.values():
        continue
    m = json.loads((p / "manifest.json").read_text())
    everything.setdefault(m["model"], p)

table = compare_runs(everything)
display(table.style.format("{:.0%}").background_gradient(cmap="Reds", vmin=0, vmax=1))

## 8. Reading it honestly

The comparison that matters is **`partner_confirmed` against `court`**. In both open base models tested so far those are within a few points of each other — the model weighs a supervising partner's assertion like a court's holding. gpt-5.1 keeps them apart and far lower.

Whichever way Thomson-1 falls, two things must not be claimed. This is one product, on 30 England-and-Wales scenarios, in a forced-choice format — not a general verdict on it. And a result here says nothing about CoCounsel as a system: a shipped product wraps the model in retrieval, prompting and guardrails this measurement never touches.

If the result is unfavourable, give Thomson Reuters notice and a right of reply before publishing. That is the norm for evaluation research naming a commercial product, and it makes the paper better rather than weaker.

## 9. Save

In [ ]:
import shutil
for label, path in runs.items():
    archive = shutil.make_archive(f"/content/{path.name}", "zip", path)
    print(archive)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as exc:
        print(f"  download from the file browser instead ({exc})")